# Fermi-Dirac Distribution Interactive Workbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# Ensure inline plotting engine is explicitly configured for the cell context
%matplotlib inline

kB = 8.617333262145e-5  # Boltzmann constant in eV/K


def fermi_dirac(E, mu, T, kB=kB):
    """
    Fermi-Dirac mean occupation number.

    f_FD(E) = 1 / (exp((E - mu)/(kB T)) + 1)

    Accepts scalars or NumPy arrays.
    """

    E = np.asarray(E)

    if T == 0:
        return np.where(E < mu, 1.0, np.where(E == mu, 0.5, 0.0))

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    return 1.0 / (np.exp(x) + 1.0)
def maxwell_boltzmann(E, mu, T, kB=kB):
    """
    Maxwell-Boltzmann dilute-limit occupation factor.

    f_MB(E) = exp(-(E - mu)/(kB T))

    This is not bounded above by 1.
    """

    E = np.asarray(E)

    if T == 0:
        return np.where(E > mu, 0.0, np.inf)

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    return np.exp(-x)


def bose_einstein(E, mu, T, kB=kB):
    """
    Bose-Einstein mean occupation number.

    f_BE(E) = 1 / (exp((E - mu)/(kB T)) - 1)

    The expression diverges as E approaches mu from above.
    For a physical ideal Bose gas, mu must be below the lowest
    single-particle energy.
    """

    E = np.asarray(E)

    if T == 0:
        return np.zeros_like(E, dtype=float)

    x = (E - mu) / (kB * T)
    x = np.clip(x, -700, 700)

    denom = np.exp(x) - 1.0

    return np.where(denom > 0, 1.0 / denom, np.nan)


plot_output = widgets.Output()


# Create a dedicated, isolated HTML rendering container for the plot engine
plot_output = widgets.Output()

def redraw_workbook(change=None):
    """
    Clears the output container and draws a fresh canvas with rigid, fixed bounds.
    """

    # Define sliders
    Ef = Ef_slider.value
    T = T_slider.value
    
    # Route all matplotlib printing behaviors directly into our isolated display box
    with plot_output:
        plot_output.clear_output(wait=True)
        
        # 1. Instantiate a localized, explicit canvas frame loop
        fig, ax = plt.subplots(figsize=(8, 5))
        
        # 2. Compute the static mesh grid array (0 to 5.5 eV)
        E_grid = np.linspace(0.0, 5.5, 1000)
        f_FD = np.array([fermi_dirac(e, Ef, T) for e in E_grid])
        f_MB = np.array([maxwell_boltzmann(e, Ef, T) for e in E_grid])

        # 3. Render vector components
        f_FD_label = 'Fermi-Dirac'
        f_FD_color = 'royalblue'
        f_FD_lw = 2.5
        f_MB_label = 'Maxwell-Boltzmann'
        f_MB_color = "darkorange"
        f_MB_lw = 2.5

        ax.plot(
            E_grid, 
            f_FD, 
            label=f_FD_label, 
            color=f_FD_color, 
            lw=f_FD_lw, 
            linestyle='--')
        ax.plot(
            E_grid, 
            f_MB, 
            label=f_MB_label, 
            color=f_MB_color, 
            lw=f_MB_lw, linestyle='--')
        ax.axvline(Ef, color='crimson', linestyle='--', lw=1.5, label=f'Fermi Level (Ef = {Ef:.1f} eV)')
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7)
        
        # 4. required to hardlock xlim and ylim
        ax.set_xlim(  0.0, 5.5)
        ax.set_ylim(-0.05, 2.05)
        
        # 5. Presentation properties

 
        ax.set_title('Fermi-Dirac Distribution Occupancy Probability', fontsize=14, pad=12)
        ax.set_xlabel('Energy, E (eV)', fontsize=12)
        ax.set_ylabel('Occupancy Probability, f(E)', fontsize=12)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
        
        # 6. Flush active frame objects down the pipeline immediately
        plt.show()

# Build interface hardware explicitly
Ef_slider = widgets.FloatSlider(
      min=0.5, 
      max=5.0, 
      step=0.1, 
      value=3.0, 
      description='Ef (eV)')
T_slider = widgets.IntSlider(
      min=0, 
      max=2000, 
      step=50, 
      value=300, 
      description='Temp (K)')

# Watch for real-time value changes to instantly kick-start the renderer loop
Ef_slider.observe(redraw_workbook, names='value')
T_slider.observe(redraw_workbook, names='value')

# Trigger the very first render state calculation so it is visible immediately upon execution
redraw_workbook()

# Package UI sliders and the output canvas neatly into a locked column layout container
widgets.VBox([Ef_slider, T_slider, plot_output])


In [ ]:
import numpy as np
import pytest

# --- Pytest Style Test Functions ---

def test_fermi_at_fermi_level():
    """At E = Ef, the occupancy probability must always be exactly 0.5."""
    assert fermi_dirac(E=3.0, Ef=3.0, T=300) == pytest.approx(0.5)


def test_fermi_excited_state():
    """At E = 3.1 eV, f(E) should be roughly 0.0205 (2.05% occupancy)."""
    expected_probability = 0.020468
    assert fermi_dirac(E=3.1, Ef=3.0, T=300) == pytest.approx(expected_probability, abs=1e-4)


def test_fermi_lower_state():
    """At E = 2.9 eV, f(E) should be roughly 0.9795 (97.95% occupancy)."""
    expected_probability = 0.979531
    assert fermi_dirac(E=2.9, Ef=3.0, T=300) == pytest.approx(expected_probability, abs=1e-4)


@pytest.mark.parametrize("E, Ef, T, expected", [
    (2.5, 3.0, 0, 1.0),  # Below Fermi level at absolute zero -> 100% full
    (3.5, 3.0, 0, 0.0),  # Above Fermi level at absolute zero -> 100% empty
    (3.0, 3.0, 0, 0.5),  # At Fermi level at absolute zero -> 50%
])
def test_absolute_zero_edge_cases(E, Ef, T, expected):
    """Ensures boundary conditions hold perfectly at 0 Kelvin."""
    assert fermi_dirac(E, Ef, T) == expected


In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# # Interactive Electronic Carrier Distribution Workbook

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# Ensure inline plotting engine is explicitly configured for the cell context
%matplotlib inline

# %%
def fermi_dirac(E, mu, T):
    """Calculates the Fermi-Dirac occupancy probability."""
    kB = 8.617333262145e-5  # eV/K
    if T == 0:
        return np.piecewise(E, [E < mu, E == mu, E > mu], [1.0, 0.5, 0.0])
    exponent = np.clip((E - mu) / (kB * T), -700, 700)
    return 1.0 / (np.exp(exponent) + 1.0)

def maxwell_boltzmann(E, mu, T):
    """Calculates the classical Maxwell-Boltzmann distribution factor."""
    kB = 8.617333262145e-5  # eV/K
    if T == 0:
        return np.where(E < mu, 1.0, np.where(E == mu, 0.5, 0.0))
    exponent = np.clip((E - mu) / (kB * T), -700, 700)
    return np.exp(-exponent)

def density_of_states(E):
    """Calculates the 3D parabolic density of states g(E) proportional to sqrt(E)."""
    # Returns 0 for negative energy states, smoothly scales with sqrt(E) above 0
    return np.where(E > 0, np.sqrt(E), 0.0)

# %%
# Create a dedicated HTML rendering container for the plot engine
plot_output = widgets.Output()

def redraw_workbook(change=None):
    Ef = Ef_slider.value
    T = T_slider.value
    
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        
        # Grid range from 0.0 eV up to 5.5 eV
        E_grid = np.linspace(0.0, 5.5, 1000)
        
        # 1. Compute physical components
        g_E = density_of_states(E_grid)
        f_FD = fermi_dirac(E_grid, Ef, T)
        f_MB = maxwell_boltzmann(E_grid, Ef, T)
        
        # 2. Compute actual electron concentration distributions: n(E) = g(E) * f(E)
        n_FD = g_E * f_FD
        n_MB = g_E * f_MB
        
        # 3. Plot the continuous physical curves
        ax.plot(E_grid, n_FD, label='g(e)f_fd(e)', color='royalblue', lw=2.5)
        ax.plot(E_grid, n_MB, label='g(e)f_mb(e)', color='darkorange', lw=2, linestyle='--')
        ax.plot(E_grid, g_E, label='g(E)', color='dimgray', lw=1.5, linestyle=':')
        
        # Reference markers
        ax.axvline(Ef, color='crimson', linestyle='--', lw=1.5, label=f'$E_f$ = {Ef:.1f} eV')
        
        # 4. HARD-LOCK AXIS VIEWPORTS
        ax.set_xlim(0.0, 5.5)
        # Scaled dynamically to state density max so curves are fully visible without clipping
        ax.set_ylim(-0.05, np.max(g_E) * 1.2)  
        
        # 5. Styling
        ax.set_title('Actual Carrier Distribution: Energy State Density vs. Temperature', fontsize=14, pad=12)
        ax.set_xlabel('Energy, E (eV)', fontsize=12)
        ax.set_ylabel('Number of Occupied States, n(E)', fontsize=12)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
        
        plt.show()

# %%
# Build interface sliders
Ef_slider = widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=3.0, description='Ef (eV)')
T_slider = widgets.IntSlider(min=50, max=2000, step=50, value=300, description='Temp (K)')

# Attach observer threads
Ef_slider.observe(redraw_workbook, names='value')
T_slider.observe(redraw_workbook, names='value')

# Trigger layout render instantly on cell load
redraw_workbook()

# Return clean widget framework configuration
widgets.VBox([Ef_slider, T_slider, plot_output])
